In [1]:
!pip install pandas plotly

In [2]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

# Read CSV file - Data Import
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))


Saving Brent_Oil_Price_2026_Weekly - GOOGLE_QUERY.csv to Brent_Oil_Price_2026_Weekly - GOOGLE_QUERY.csv


In [3]:
# Representation of dataframe data and verification data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    44 non-null     object 
 1   Open    44 non-null     float64
 2   High    44 non-null     float64
 3   Low     44 non-null     float64
 4   Close   44 non-null     float64
 5   Volume  44 non-null     int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 2.2+ KB


In [4]:
# Code to convert the data type of Date from string to datetime
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

In [5]:
df.head()

,Date,Open,High,Low,Close,Volume
0,2026-01-02 23:58:00,60.58,61.91,59.53,60.24,100312
1,2026-01-09 23:58:00,59.93,62.63,59.19,62.29,111404
2,2026-01-16 23:58:00,62.77,65.20,61.74,62.80,196441
3,2026-01-23 23:58:00,62.80,64.46,62.07,64.24,123887
4,2026-01-30 23:58:00,63.97,69.09,63.60,68.04,208674


In [6]:
df.tail()

,Date,Open,High,Low,Close,Volume
39,2026-10-02 23:58:00,98.50,103.05,95.14,102.25,293032
40,2026-10-09 23:58:00,103.01,105.91,97.08,104.72,198300
41,2026-10-16 23:58:00,104.64,110.00,102.35,109.86,188502
42,2026-10-23 23:58:00,110.25,117.41,105.95,116.72,181833
43,2026-10-27 23:58:00,115.80,115.96,113.91,113.97,0


In [7]:
# Data import summary
print("Nombre de Archivo:", filename)
print("Número de filas:", len(df))
print("Periodo:", df["Date"].min().date(), "a", df["Date"].max().date())

Nombre de Archivo: Brent_Oil_Price_2026_Weekly - GOOGLE_QUERY.csv
Número de filas: 44
Periodo: 2026-01-02 a 2026-10-27


In [8]:
# Verify OHLC conditions for price value are achieved
valid_ohlc = (
    (df["Low"] <= df["Open"]) &
    (df["Open"] <= df["High"]) &
    (df["Low"] <= df["Close"]) &
    (df["Close"] <= df["High"])
)

print(
    "Filas que verifican condiciones del gráfico OHLC:", valid_ohlc.sum(),
    "de", len(df)
)

# Verify rows with volume = 0
print("Filas con volumen cero:")
display(df.loc[df["Volume"] == 0])

Filas que verifican condiciones del gráfico OHLC: 44 de 44
Filas con volumen cero:


,Date,Open,High,Low,Close,Volume
43,2026-10-27 23:58:00,115.8,115.96,113.91,113.97,0


In [9]:
# Filter rows with volume = 0
import plotly.graph_objects as go
from IPython.display import display, HTML

df_plot = df.loc[df["Volume"] > 0].copy()

# Ensure data is ordered in ascendent order by Date
df_plot = df_plot.sort_values("Date")

# Background and grid colors
plot_background = "#F1F5F9"
paper_background = "#FFFFFF"
grid_color = "#D8E0E8"

# Reference 1: https://plotly.com/python/ohlc-charts/
# Reference 2: ChatGPT Prompt: "Review the code to support an OHLC chart to
# represent the Brent Oil pirce per Barrel during 2026 in weekly periods"
# Plot configuration
fig = go.Figure(
    data=[
        go.Ohlc(
            x=df_plot["Date"],
            open=df_plot["Open"],
            high=df_plot["High"],
            low=df_plot["Low"],
            close=df_plot["Close"],
            name="Brent",
            increasing_line_color="#16805D",
            decreasing_line_color="#C44536",
            line_width=2,
            hoverlabel=dict(namelength=-1)
        )
    ]
)

# Adding annotation on the start of Iran's War conflict
event_date = pd.Timestamp("2026-02-28")

fig.add_vline(
    x=event_date,
    line_width=1,
    line_dash="solid",
    line_color="#64748B",
    opacity=0.6
)

fig.update_layout(
    title=dict(
        text="Evolución semanal de los futuros del barril de Brent en 2026"
        " y el efecto de la Guerra de Irán",
        x=0.05,
        xanchor="left"
    ),
    xaxis_title="Fecha",
    yaxis_title="Precio (USD por barril)",

    template="plotly_white",
    paper_bgcolor=paper_background,
    plot_bgcolor=plot_background,
    height=700,
    hovermode="x",
    margin=dict(l=75, r=35, t=100, b=70),
    font=dict(family="Arial", size=13, color="#263445"),
    xaxis=dict(
        type="date",
        showgrid=True,
        gridcolor=grid_color,
        showspikes=True,
        spikemode="across",
        spikesnap="cursor",
        spikecolor="#475569",
        spikethickness=1.5,
        spikedash="solid",

        rangeslider=dict(
            visible=True,
            bgcolor="#E2EAF2",
            bordercolor="#CBD5E1",
            borderwidth=1,
            thickness=0.12
        ),

        rangeselector=dict(
            bgcolor="#E2EAF2",
            activecolor="#CBD5E1",
            buttons=[
                dict(count=1, label="1 mes", step="month",
                     stepmode="backward"),
                dict(count=3, label="3 meses", step="month",
                     stepmode="backward"),
                dict(count=6, label="6 meses", step="month",
                     stepmode="backward"),
                dict(label="Año 2026", step="all")
            ]
        )
    ),

    yaxis=dict(
        tickprefix="$",
        showgrid=True,
        gridcolor=grid_color,
        zeroline=False
    ),

    annotations=[
        dict(
            text="Fuente: Google Sheets - GOOGLEFINANCE — NYMEX:BZW00",
            xref="paper",
            yref="paper",
            x=0,
            y=-0.3,
            showarrow=False,
            xanchor="left",
            font=dict(size=12, color="#555555")
        ),
        dict(
            text="Autor: Victor Calvo Serra",
            xref="paper",
            yref="paper",
            x=0,
            y=-0.35,
            showarrow=False,
            xanchor="left",
            font=dict(size=12, color="#555555")
        ),
        dict(
            x=event_date,
            y=1,
            text="<b>Inicio de la Guerra de Irán</b>",
            xref="x",
            yref="paper",
            showarrow=False,
            xanchor="left",
            yanchor="top",
            xshift=7,
            yshift=-5,
            font=dict(size=12, color="#555555")
        )
    ]
)

fig.show()

# Analytical comment between chart and data table
# Reference 1: ChatGPT Prompt: "Review the code to display html comments
# for a description to explain main features for an OHLC chart. Text in
# Spanish"
commentary_html = """
<div class="commentary">
    <h3>Claves para interpretar el gráfico</h3>
    <p>
        El gráfico muestra la evolución semanal de los futuros del barril de
        petróleo Brent durante el año 2026. Cada barra representa el precio
        de apertura, el máximo, el mínimo y el cierre de una semana. Las barras
        verdes indican semanas en las que el precio de cierre superó al de
        apertura; las rojas indican lo contrario.
    </p>
    <p>
        La línea vertical señala el 28 de febrero de 2026, fecha de referencia
        del inicio de la guerra con Irán. Se observa una evolución creciente
        significativa de los precios las dos semanas siguientes. Entre el 20
        de marzo y el 15 de mayo los precios de cierre oscilan entre 90 y 114 $,
        sin embargo se puede observar una gran volatilidad del mercado debido a
        la situación inestable. La semana del 22 de mayo hubo una fuerte caída y
        las semanas siguientes muestran una reducción de cierres de precio y
        de los rangos. Finalmente, desde el 10 de julio y hasta octubre el
        precio de cierre ha ido creciendo, pero los rangos de precios semanales
        son mucho más pequeños que en marzo de 2026, indicando un crecimiento
        con menos volatilidad y mucho más estable.

        A continuación, se presentan los datos usados para la visualización.
    </p>
</div>
"""

display(HTML("""
<style>
.commentary {
    background: #FFFFFF;
    border-left: 4px solid #64748B;
    border-radius: 5px;
    padding: 12px 18px;
    margin: 18px 0 24px 0;
    color: #263445;
    line-height: 1.6;
    max-width: 1100px;
}
.commentary h3 {
    margin-top: 0;
    margin-bottom: 8px;
    font-size: 18px;
}
.commentary p {
    margin: 8px 0;
}
</style>
""" + commentary_html))


# Present table of the data in the html file
table_df = df_plot[
    ["Date", "Open", "High", "Low", "Close"]
].copy()

table_df["Date"] = table_df["Date"].dt.strftime("%d-%m-%Y")
for col in ["Open", "High", "Low", "Close"]:
    table_df[col] = table_df[col].round(2)

display(HTML(
    "<h3>Datos utilizados en el gráfico</h3>"
    + table_df.to_html(
        index=False,
        classes="data-table",
        border=0
    )
))


Date,Open,High,Low,Close
02-01-2026,60.58,61.91,59.53,60.24
09-01-2026,59.93,62.63,59.19,62.29
16-01-2026,62.77,65.20,61.74,62.80
23-01-2026,62.80,64.46,62.07,64.24
30-01-2026,63.97,69.09,63.60,68.04
06-02-2026,66.35,68.36,64.13,67.00
13-02-2026,66.35,69.30,65.96,66.60
20-02-2026,66.52,71.09,65.76,70.66
27-02-2026,70.66,73.00,68.80,72.29
06-03-2026,76.04,87.66,74.64,87.20


In [10]:
# Code to generate the html objects for the chart and table
chart_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True
)

table_html = table_df.to_html(
    index=False,
    border=0,
    classes="data-table"
)

In [11]:
# Code to onfigure the html page
# Reference 1: ChatGPT Prompt: "Review the html code to display three objects
# and show a correct format"
page_html = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Brent Oil — OHLC 2026</title>
<style>
    body {{
        margin: 0;
        padding: 24px;
        font-family: Arial, sans-serif;
        background: #FFFFFF;
        color: #263445;
    }}
    main {{
        max-width: 1400px;
        margin: auto;
    }}
    .commentary {{
        background: #F1F5F9;
        border-left: 4px solid #64748B;
        border-radius: 5px;
        padding: 12px 18px;
        margin: 18px 0 24px 0;
        line-height: 1.6;
    }}
    .commentary h3 {{
        margin-top: 0;
        margin-bottom: 8px;
        font-size: 18px;
    }}
    .commentary p {{
        margin: 8px 0;
    }}
    .table-container {{
        overflow-x: auto;
        max-height: 420px;
        overflow-y: auto;
        background: white;
        border: 1px solid #D8E0E8;
        border-radius: 8px;
        margin-top: 18px;
    }}
    .data-table {{
        border-collapse: collapse;
        width: 100%;
        font-size: 13px;
    }}
    .data-table th {{
        position: sticky;
        top: 0;
        background: #E2EAF2;
        text-align: left;
    }}
    .data-table th, .data-table td {{
        padding: 10px 12px;
        border-bottom: 1px solid #E5EAF0;
        white-space: nowrap;
    }}
    .data-table tbody tr:nth-child(even) {{
        background: #F8FAFC;
    }}
    .source {{
        font-size: 12px;
        color: #526173;
        margin-top: 24px;
    }}
</style>
</head>
<body>
<main>
    <h1>Evolución semanal de los futuros del barril de Brent en 2026
    y el efecto de la guerra de Irán</h1>

    {chart_html}

    {commentary_html}

    <h2>Datos utilizados en el gráfico</h2>
    <p>Observaciones semanales representadas en el gráfico.</p>
    <div class="table-container">
        {table_html}
    </div>

    <p class="source">
        Fuente: Google Sheets — GOOGLEFINANCE (NYMEX:BZW00).
        Periodicidad semanal. Precios en USD por barril.
    </p>
</main>
</body>
</html>
"""

from pathlib import Path

Path("index.html").write_text(page_html, encoding="utf-8")
print("index.html generado con el gráfico, el comentario y la tabla.")

index.html generado con el gráfico, el comentario y la tabla.


In [12]:
files.download("index.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>